# Phase 9 (completion) — N=50 fill-ins, Tier 2 + Tier 3
Single seed (42) each, since Tier 2/3 already have solid 3-seed evidence at N=10 — this checks whether the pattern holds with more training data, not another full seed sweep. 4 runs total: Tier2 DSPy, Tier2 QLoRA, Tier3 DSPy(guarded), Tier3 QLoRA.

Same 4-stage, restart-between-stages structure as the N=10 notebook.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show 0MiB used before continuing.**

## Setup (repeat this after every restart)

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pip install -q transformers accelerate bitsandbytes peft datasets dspy-ai optuna

In [ ]:
from huggingface_hub import login
login()

---
## STAGE E — Tier 2 DSPy, N=50, seed=42

In [ ]:
import sys, json, gc, torch
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier2_training
from tasks.tier2 import TIER2_HELDOUT
import dspy_optimize_tier2

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model ready.")

In [ ]:
gc.collect(); torch.cuda.empty_cache()
train_50 = sample_tier2_training(50, seed=42)
print(f"Training on {len(train_50)} Tier 2 examples (N=50).")

optimized = dspy_optimize_tier2.optimize(lm, train_50, seed=42)
results = dspy_optimize_tier2.evaluate_program(optimized, TIER2_HELDOUT)
rate = sum(r["grade"]["success"] for r in results) / len(results)
print(f"Tier 2 DSPy N=50 seed=42: {rate:.1%}")
with open("results/tier2_dspy_n50_seed42_results.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
files.download("results/tier2_dspy_n50_seed42_results.json")

**Download, push, then RESTART before Stage F.**

---
## STAGE F — Tier 3 DSPy (guarded), N=50, seed=42
Re-run Setup first.

In [ ]:
import sys, json, gc, torch
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier3_training
from tasks.tier3 import TIER3_HELDOUT
import dspy_optimize_tier3

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model ready.")

In [ ]:
gc.collect(); torch.cuda.empty_cache()
train_50 = sample_tier3_training(50, seed=42)
print(f"Training on {len(train_50)} Tier 3 examples (N=50).")

optimized = dspy_optimize_tier3.optimize(lm, train_50, guarded=True, seed=42)
results = dspy_optimize_tier3.evaluate_program(optimized, TIER3_HELDOUT)
rate = sum(r["grade"]["success"] for r in results) / len(results)
print(f"Tier 3 DSPy (guarded) N=50 seed=42: {rate:.1%}")
with open("results/tier3_dspy_guarded_n50_seed42_results.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
files.download("results/tier3_dspy_guarded_n50_seed42_results.json")

**Download, push, then RESTART before Stage G.**

---
## STAGE G — Tier 2 QLoRA, N=50, seed=42
Re-run Setup first.

In [ ]:
!python qlora_finetune_tier2.py --n 50 --seed 42

In [ ]:
import torch, json, gc, sys
sys.path.insert(0, ".")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from tasks.tier2 import TIER2_HELDOUT
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map={"": 0})
ft_model = PeftModel.from_pretrained(base_model, "adapters/tier2_n50_seed42")
ft_tok = AutoTokenizer.from_pretrained("adapters/tier2_n50_seed42")

results = []
for task in TIER2_HELDOUT:
    tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=2, tool_calls=tool_calls, final_text=final_text)
    results.append({"id": task["id"], "prompt": task["prompt"], "tool_calls": tool_calls, "final_text": final_text, "grade": grade})
rate = sum(r["grade"]["success"] for r in results) / len(results)
print(f"Tier 2 QLoRA N=50 seed=42: {rate:.1%}")
with open("results/tier2_qlora_n50_seed42_results.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
files.download("results/tier2_qlora_n50_seed42_results.json")

**Download, push, then RESTART before Stage H.**

---
## STAGE H — Tier 3 QLoRA, N=50, seed=42
Re-run Setup first.

In [ ]:
!python qlora_finetune_tier3.py --n 50 --seed 42

In [ ]:
import torch, json, gc, sys
sys.path.insert(0, ".")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from tasks.tier3 import TIER3_HELDOUT
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map={"": 0})
ft_model = PeftModel.from_pretrained(base_model, "adapters/tier3_n50_seed42")
ft_tok = AutoTokenizer.from_pretrained("adapters/tier3_n50_seed42")

results = []
for task in TIER3_HELDOUT:
    tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=3, tool_calls=tool_calls, final_text=final_text)
    results.append({"id": task["id"], "prompt": task["prompt"], "tool_calls": tool_calls, "final_text": final_text, "grade": grade})
rate = sum(r["grade"]["success"] for r in results) / len(results)
print(f"Tier 3 QLoRA N=50 seed=42: {rate:.1%}")
with open("results/tier3_qlora_n50_seed42_results.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
files.download("results/tier3_qlora_n50_seed42_results.json")

## Done — Phase 9 fully complete after this.
No aggregation script needed here (single seed each) — just check the 4 printed percentages directly against the N=10 numbers to see whether the pattern holds, strengthens, or changes with more data.